# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [2]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
pd.set_option('display.max_columns', None)

## Lottery and Sales Data (for reference when merging)

In [22]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [23]:
locations_df.head(2)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid
0,55194,BOWL-O-DROME,2712 PINE AVE,NIAGARA FALLS,14301,0,BOWLING CENTERS,-79.02878,43.09513,b'\x01\x01\x00\x00\x00\xd9\xb1\x11\x88\xd7\xc1...,360630210002,36063021000
1,68523,YAHIR & SISTERS GROCERY CORP,165 MCLEAN AVE,YONKERS,10705,0,GROCERY STORE,-73.88996,40.91546,b'\x01\x01\x00\x00\x00\xe2\xaf\xc9\x1a\xf5xR\x...,361190013013,36119001301


In [24]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [25]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
12455,78444,22509.0,2025-05-16,2025-09-25,2025-05-16,2025-09-25,74.0,133,133.0,0.0,0.0,169.240602,169.240602,False,169.240602
1832,106285,1118313.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,363.0,365,365.0,0.0,0.0,3063.872603,3063.872603,False,3063.872603
4224,111893,122635.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,364.0,365,365.0,0.0,0.0,335.987671,335.987671,False,335.987671
4329,112097,8282.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,139.0,365,365.0,0.0,0.0,22.691781,22.691781,False,22.691781
11713,74837,63000.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,310.0,365,365.0,0.0,0.0,172.604110,172.604110,False,172.604110


## Overture Maps Data

In [28]:
overture_file_path = Path(".").resolve().parent / "data/final/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [30]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi
81,108868,WEST44TH STREET NEWSSTAND LLC,300 WEST 44TH STREET,NEW YORK,10036,-73.98944,40.75879,b'\x01\x01\x00\x00\x00\xacV&\xfcR\x7fR\xc0\x13...,360610121012,36061012101,5,6,50,130,44,137,127,423,71,187,54,129
79,76831,ALBANY NEWS DELI,396 CENTRAL AVE,ALBANY,12206,-73.77589,42.66720,b'\x01\x01\x00\x00\x00\xc8\xd2\x87.\xa8qR\xc0\...,360010005011,36001000501,2,5,1,2,1,2,5,39,3,3,2,3
43,111327,FAIRFIELD FOOD & FUEL CORP,182 FAIRFIELD DR,BREWSTER,10509,-73.54064,41.45572,b'\x01\x01\x00\x00\x004\xba\x83\xd8\x99bR\xc0\...,360790101001,36079010100,1,1,0,0,0,0,0,0,0,0,0,0
34,76110,MARINELLI'S VILLAGE PHARMACY,31 E MAIN ST,ELMSFORD,10523,-73.81836,41.05442,b'\x01\x01\x00\x00\x00\xb1\x16\x9f\x02`tR\xc0\...,361190112004,36119011200,5,9,1,1,3,5,3,5,0,1,0,0
76,75465,BARREL INN II LTD,208 EAST MEADOW AVE,EAST MEADOW,11554,-73.56331,40.71864,b'\x01\x01\x00\x00\x00\xa0\xe0bE\rdR\xc0y\xcc@...,360594080001,36059408000,6,8,0,0,2,5,1,5,0,2,1,2


## LODES Data

In [10]:
rac_file_path = Path(".").resolve().parent / "data/final/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [11]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
92,110788,SUNRISE 99 CENT STORE INC,4008 QUEENS BLVD,SUNNYSIDE,11104,-73.92392,40.74361,b'\x01\x01\x00\x00\x00\xb3\x07Z\x81!{R\xc0A}\x...,360810181013,36081018101,738.0,117.0,182.0,439.0
81,76831,ALBANY NEWS DELI,396 CENTRAL AVE,ALBANY,12206,-73.77589,42.66720,b'\x01\x01\x00\x00\x00\xc8\xd2\x87.\xa8qR\xc0\...,360010005011,36001000501,440.0,104.0,146.0,190.0


In [12]:
wac_file_path = Path(".").resolve().parent / "data/final/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [13]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
40,68056,7-ELEVEN #16361E,33 MIDDLE COUNTRY RD RTE 25,CORAM,11727,-72.97863,40.87459,b'\x01\x01\x00\x00\x00\x9b8\xb9\xdf\xa1>R\xc0N...,361031583091,36103158309,209.0,36.0,4.0,37.0,68.0,104.0
52,107828,GATEWAY NEWSSTAND #655,1849 2ND AVE,NEW YORK,10128,-73.94840,40.78408,b'\x01\x01\x00\x00\x00\xb1\xe1\xe9\x95\xb2|R\x...,360610156014,36061015601,266.0,11.0,0.0,29.0,41.0,196.0


## AADT Data

In [15]:
aadt_file_path = Path(".").resolve().parent / "data/final/data_aadt_2025.parquet"

aadt_pq = pq.ParquetFile(aadt_file_path)
aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

In [19]:
aadt_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,bg_geoid,tract_geoid,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m
16,45101,SPEEDWAY #07665,1511 BUFFALO RD,ROCHESTER,14624,-77.69117,43.14956,360550144004,36055014400,55617.0,75037.0,NY390,12,2:2||||,1071.058866
83,113999,TRIPPLE STAR LIQUOR & WINE INC,2025 3RD AVE,NEW YORK,10029,-73.94228,40.79486,360610172001,36061017200,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1206.716356
9,106285,7 ELEVEN STORE #35195A,2300 E MAIN ST,ROCHESTER,14609,-77.54874,43.15974,360550061002,36055006100,55612.0,77615.0,NaN,12,2:2|3:2|3:3||,1357.599431
33,114951,HARLEM'S OWN WINE AND LIQUOR L,314 E 106TH ST,NEW YORK,10029,-73.94078,40.78993,360610170004,36061017000,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1540.238987
66,92154,LEXINGTON WINE & LIQUOR,2010 LEXINGTON AVE,NEW YORK,10035,-73.93851,40.80288,360610196003,36061019600,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,628.511476


## EPA Data

In [20]:
epa_file_path = Path(".").resolve().parent / "data/interim/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [21]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,GEOID10,GEOID20,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,CBSA_POP,CBSA_EMP,HH,D3B,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
5,77603,SUPER D,357 N MAIN ST,FREEPORT,11520,-73.58764,40.66898,b'\x01\x01\x00\x00\x00\x92t\xcd\xe4\x9beR\xc0\...,360594141005,36059414100,157336,3.605940e+11,3.605940e+11,36,59,414100,5,19318471,9159786,327.0,185.123984,15.500000,36,059,414100,5
93,77429,RUNWAY #6371,6371 ROUTE 167,DOLGEVILLE,13329,-74.78544,43.09375,"b'\x01\x01\x00\x00\x00\x18>""\xa6D\xb2R\xc0\x00...",360430110023,36043011002,157376,3.604300e+11,3.604300e+11,36,43,11002,3,293287,122479,361.0,0.922333,5.333333,36,043,011002,3
55,37654,APPLE FOOD & GROCERY INC,340 MAIN ST,MIDDLEBURGH,12122,-74.33595,42.59917,b'\x01\x01\x00\x00\x00\xd6\xc5m4\x80\x95R\xc0F...,360957407004,36095740700,149708,3.609570e+11,3.609570e+11,36,95,740700,4,880481,447103,557.0,1.565185,5.000000,36,095,740700,4
13,100075,965 A & A DELI GROCERY CORP,965 MORRIS PARK AVE,BRONX,10462,-73.85730,40.84763,b'\x01\x01\x00\x00\x00\x17\xb7\xd1\x00\xdevR\x...,360050244002,36005024400,161219,3.600500e+11,3.600500e+11,36,5,24400,2,19318471,9159786,431.0,98.548492,14.500000,36,005,024400,2
35,111759,TT2 TRADING INC,376 S BAYVIEW AVE,FREEPORT,11520,-73.59283,40.64398,b'\x01\x01\x00\x00\x00\x9d\x85=\xed\xf0eR\xc0\...,360594145012,36059414501,157438,3.605940e+11,3.605940e+11,36,59,414501,2,19318471,9159786,487.0,183.901022,19.000000,36,059,414501,2
